In [45]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/wesad_readme.pdf
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14.pkl
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_quest.csv
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_respiban.txt
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_readme.txt
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_E4_Data/HR.csv
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_E4_Data/IBI.csv
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_E4_Data/BVP.csv
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_E4_Data/EDA.csv
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-da

In [46]:
!pip install neurokit2 catboost -q

In [47]:
import os
import pickle
import json
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy import signal
from scipy.stats import median_abs_deviation

import neurokit2 as nk

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [48]:
# ============================================================
# CONFIGURATION
# ============================================================

BASE_PATH = (
    "/kaggle/input/datasets/orvile/"
    "wesad-wearable-stress-affect-detection-dataset/"
    "WESAD"
)

# ------------------------------------------------------------
# Sampling frequencies - WESAD wrist device
# ------------------------------------------------------------

EDA_FS = 4
BVP_FS = 64
ACC_FS = 32

# WESAD label signal is 700 Hz
LABEL_FS = 700

# ------------------------------------------------------------
# Window configuration
# ------------------------------------------------------------

WINDOW_SECONDS = 30
STEP_SECONDS = 15

# ------------------------------------------------------------
# Baseline configuration
# ------------------------------------------------------------

BASELINE_LABEL = 1

# Minimum percentage of baseline samples
# required for a window to be considered baseline.
BASELINE_PURITY = 0.90

# ------------------------------------------------------------
# ML configuration
# ------------------------------------------------------------

RANDOM_STATE = 42

print("Configuration loaded.")

Configuration loaded.


In [49]:
# ============================================================
# DISCOVER SUBJECTS
# ============================================================

subjects = sorted([
    s for s in os.listdir(BASE_PATH)
    if s.startswith("S")
    and os.path.isdir(os.path.join(BASE_PATH, s))
])

print("Subjects:")
print(subjects)

print("\nNumber of subjects:", len(subjects))

Subjects:
['S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']

Number of subjects: 15


In [50]:
# ============================================================
# WESAD SUBJECT LOADER
# ============================================================

def load_subject(subject_id):

    subject_path = os.path.join(
        BASE_PATH,
        subject_id,
        f"{subject_id}.pkl"
    )

    if not os.path.exists(subject_path):
        raise FileNotFoundError(
            f"File not found: {subject_path}"
        )

    with open(subject_path, "rb") as f:
        data = pickle.load(
            f,
            encoding="latin1"
        )

    wrist = data["signal"]["wrist"]

    eda = np.asarray(
        wrist["EDA"]
    ).flatten()

    bvp = np.asarray(
        wrist["BVP"]
    ).flatten()

    acc = np.asarray(
        wrist["ACC"]
    )

    labels = np.asarray(
        data["label"]
    ).flatten()

    return {
        "eda": eda,
        "bvp": bvp,
        "acc": acc,
        "labels": labels
    }

In [51]:
data = load_subject("S2")

for key, value in data.items():
    print(
        key,
        value.shape,
        value.dtype
    )

eda (24316,) float64
bvp (389056,) float64
acc (194528, 3) float64
labels (4255300,) int32


In [52]:
# ============================================================
# BASIC SIGNAL QUALITY CHECK
# ============================================================

def signal_quality_report(data):

    report = {}

    for name, x in data.items():

        x = np.asarray(x)

        report[name] = {
            "samples": len(x),
            "nan": int(np.isnan(x).sum()),
            "inf": int(np.isinf(x).sum()),
            "min": float(np.nanmin(x)),
            "max": float(np.nanmax(x)),
            "mean": float(np.nanmean(x)),
            "std": float(np.nanstd(x))
        }

    return pd.DataFrame(report).T


signal_quality_report(data)

,samples,nan,inf,min,max,mean,std
eda,24316.0,0.0,0.0,0.045113,1.717419,0.391743,0.329229
bvp,389056.0,0.0,0.0,-873.670000,988.080000,-0.000427,75.871236
acc,194528.0,0.0,0.0,-128.000000,127.000000,19.428199,31.163864
labels,4255300.0,0.0,0.0,0.000000,7.000000,1.212370,1.606596


In [53]:
# ============================================================
# WINDOW GENERATOR
# ============================================================

def generate_window_indices(
    duration_seconds,
    window_seconds=WINDOW_SECONDS,
    step_seconds=STEP_SECONDS
):

    starts = np.arange(
        0,
        duration_seconds - window_seconds + 1e-9,
        step_seconds
    )

    for start in starts:

        end = start + window_seconds

        yield start, end

In [54]:
# ============================================================
# SLICE SIGNAL BY TIME
# ============================================================

def slice_signal(
    signal_data,
    fs,
    start_time,
    end_time
):

    start_idx = int(
        round(start_time * fs)
    )

    end_idx = int(
        round(end_time * fs)
    )

    return signal_data[
        start_idx:end_idx
    ]

In [55]:
# ============================================================
# WINDOW LABEL
# ============================================================

def get_window_label(
    labels,
    start_time,
    end_time
):

    start_idx = int(
        round(start_time * LABEL_FS)
    )

    end_idx = int(
        round(end_time * LABEL_FS)
    )

    window_labels = labels[
        start_idx:end_idx
    ]

    if len(window_labels) == 0:
        return None, 0.0

    values, counts = np.unique(
        window_labels,
        return_counts=True
    )

    dominant_index = np.argmax(counts)

    dominant_label = int(
        values[dominant_index]
    )

    purity = (
        counts[dominant_index]
        /
        len(window_labels)
    )

    return dominant_label, purity

In [56]:
# ============================================================
# EDA FEATURES
# ============================================================

def extract_eda_features(eda_window):

    features = {
        "eda_mean": np.nan,
        "eda_std": np.nan,
        "eda_min": np.nan,
        "eda_max": np.nan,
        "eda_range": np.nan,
        "eda_slope": np.nan,
        "scl_mean": np.nan,
        "phasic_mean": np.nan,
        "phasic_std": np.nan,
        "scr_count": np.nan,
        "scr_amp_mean": np.nan,
        "scr_rise_mean": np.nan,
        "scr_recovery_mean": np.nan
    }

    try:

        eda_window = np.asarray(
            eda_window,
            dtype=float
        )

        if len(eda_window) < EDA_FS * 10:
            return features

        cleaned = nk.eda_clean(
            eda_window,
            sampling_rate=EDA_FS
        )

        decomposition = nk.eda_phasic(
            cleaned,
            sampling_rate=EDA_FS
        )

        tonic = np.asarray(
            decomposition["EDA_Tonic"]
        )

        phasic = np.asarray(
            decomposition["EDA_Phasic"]
        )

        _, peak_info = nk.eda_peaks(
            phasic,
            sampling_rate=EDA_FS
        )

        scr_peaks = peak_info.get(
            "SCR_Peaks",
            []
        )

        scr_count = len(scr_peaks)

        # ----------------------------------------------------
        # SCR amplitude
        # ----------------------------------------------------

        scr_amplitudes = peak_info.get(
            "SCR_Amplitude",
            []
        )

        if len(scr_amplitudes) > 0:
            scr_amp_mean = np.nanmean(
                scr_amplitudes
            )
        else:
            scr_amp_mean = 0.0

        # ----------------------------------------------------
        # Rise time
        # ----------------------------------------------------

        rise_times = peak_info.get(
            "SCR_RiseTime",
            []
        )

        if len(rise_times) > 0:
            scr_rise_mean = np.nanmean(
                rise_times
            )
        else:
            scr_rise_mean = 0.0

        # ----------------------------------------------------
        # Recovery
        # ----------------------------------------------------

        recovery_times = peak_info.get(
            "SCR_RecoveryTime",
            []
        )

        if len(recovery_times) > 0:
            scr_recovery_mean = np.nanmean(
                recovery_times
            )
        else:
            scr_recovery_mean = 0.0

        # ----------------------------------------------------
        # Slope
        # ----------------------------------------------------

        x = np.arange(
            len(cleaned)
        )

        slope = np.polyfit(
            x,
            cleaned,
            1
        )[0]

        features = {
            "eda_mean": np.nanmean(cleaned),
            "eda_std": np.nanstd(cleaned),
            "eda_min": np.nanmin(cleaned),
            "eda_max": np.nanmax(cleaned),
            "eda_range": (
                np.nanmax(cleaned)
                -
                np.nanmin(cleaned)
            ),
            "eda_slope": slope,
            "scl_mean": np.nanmean(tonic),
            "phasic_mean": np.nanmean(phasic),
            "phasic_std": np.nanstd(phasic),
            "scr_count": scr_count,
            "scr_amp_mean": scr_amp_mean,
            "scr_rise_mean": scr_rise_mean,
            "scr_recovery_mean": scr_recovery_mean
        }

        return features

    except Exception:
        return features

In [57]:
# ============================================================
# BVP / HR / HRV FEATURES
# ============================================================

def extract_ppg_features(bvp_window):

    features = {
        "hr": np.nan,
        "hr_std": np.nan,
        "rmssd": np.nan,
        "sdnn": np.nan,
        "pnn50": np.nan,
        "ibi_mean": np.nan,
        "ibi_std": np.nan,
        "ibi_min": np.nan,
        "ibi_max": np.nan,
        "peak_count": np.nan
    }

    try:

        bvp_window = np.asarray(
            bvp_window,
            dtype=float
        )

        if len(bvp_window) < BVP_FS * 10:
            return features

        # ----------------------------------------------------
        # Clean BVP
        # ----------------------------------------------------

        cleaned = nk.ppg_clean(
            bvp_window,
            sampling_rate=BVP_FS
        )

        # ----------------------------------------------------
        # Detect peaks
        # ----------------------------------------------------

        _, info = nk.ppg_peaks(
            cleaned,
            sampling_rate=BVP_FS
        )

        peaks = np.asarray(
            info["PPG_Peaks"]
        )

        if len(peaks) < 4:
            return features

        # ----------------------------------------------------
        # Inter-beat intervals
        # ----------------------------------------------------

        ibi_samples = np.diff(peaks)

        ibi_seconds = (
            ibi_samples / BVP_FS
        )

        ibi_ms = (
            ibi_seconds * 1000
        )

        # Remove implausible intervals
        valid = (
            (ibi_ms >= 300)
            &
            (ibi_ms <= 2000)
        )

        ibi_ms = ibi_ms[valid]

        if len(ibi_ms) < 3:
            return features

        # ----------------------------------------------------
        # Heart rate
        # ----------------------------------------------------

        hr_values = (
            60000 / ibi_ms
        )

        hr = np.median(
            hr_values
        )

        hr_std = np.std(
            hr_values
        )

        # ----------------------------------------------------
        # HRV
        # ----------------------------------------------------

        diff_ibi = np.diff(
            ibi_ms
        )

        rmssd = np.sqrt(
            np.mean(
                diff_ibi ** 2
            )
        )

        sdnn = np.std(
            ibi_ms,
            ddof=1
        )

        pnn50 = (
            np.sum(
                np.abs(diff_ibi) > 50
            )
            /
            len(diff_ibi)
            *
            100
        )

        features = {
            "hr": hr,
            "hr_std": hr_std,
            "rmssd": rmssd,
            "sdnn": sdnn,
            "pnn50": pnn50,
            "ibi_mean": np.mean(ibi_ms),
            "ibi_std": np.std(ibi_ms),
            "ibi_min": np.min(ibi_ms),
            "ibi_max": np.max(ibi_ms),
            "peak_count": len(peaks)
        }

        return features

    except Exception:
        return features

In [58]:
# ============================================================
# ACC FEATURES
# ============================================================

def extract_imu_features(acc_window):

    features = {
        "acc_mean": np.nan,
        "acc_std": np.nan,
        "acc_energy": np.nan,
        "acc_magnitude_mean": np.nan,
        "acc_magnitude_std": np.nan,
        "acc_magnitude_energy": np.nan
    }

    try:

        acc = np.asarray(
            acc_window,
            dtype=float
        )

        if acc.ndim != 2:
            return features

        if acc.shape[1] != 3:
            return features

        # ----------------------------------------------------
        # Magnitude
        # ----------------------------------------------------

        magnitude = np.sqrt(
            np.sum(
                acc ** 2,
                axis=1
            )
        )

        features = {
            "acc_mean":
                np.mean(acc),

            "acc_std":
                np.std(acc),

            "acc_energy":
                np.mean(acc ** 2),

            "acc_magnitude_mean":
                np.mean(magnitude),

            "acc_magnitude_std":
                np.std(magnitude),

            "acc_magnitude_energy":
                np.mean(magnitude ** 2)
        }

        return features

    except Exception:
        return features

In [59]:
# ============================================================
# BUILD SUBJECT FEATURE DATASET
# ============================================================

def build_subject_dataset(
    subject_id,
    baseline_only=False
):

    data = load_subject(
        subject_id
    )

    eda = data["eda"]
    bvp = data["bvp"]
    acc = data["acc"]
    labels = data["labels"]

    # --------------------------------------------------------
    # Determine common duration
    # --------------------------------------------------------

    durations = [
        len(eda) / EDA_FS,
        len(bvp) / BVP_FS,
        len(acc) / ACC_FS,
        len(labels) / LABEL_FS
    ]

    duration = min(
        durations
    )

    rows = []

    # --------------------------------------------------------
    # Generate synchronized windows
    # --------------------------------------------------------

    for start, end in generate_window_indices(
        duration
    ):

        label, purity = get_window_label(
            labels,
            start,
            end
        )

        if label is None:
            continue

        # ----------------------------------------------------
        # Baseline mode
        # ----------------------------------------------------

        if baseline_only:

            if label != BASELINE_LABEL:
                continue

            if purity < BASELINE_PURITY:
                continue

        # ----------------------------------------------------
        # Slice signals
        # ----------------------------------------------------

        eda_window = slice_signal(
            eda,
            EDA_FS,
            start,
            end
        )

        bvp_window = slice_signal(
            bvp,
            BVP_FS,
            start,
            end
        )

        acc_window = slice_signal(
            acc,
            ACC_FS,
            start,
            end
        )

        # ----------------------------------------------------
        # Feature extraction
        # ----------------------------------------------------

        features = {}

        features.update(
            extract_eda_features(
                eda_window
            )
        )

        features.update(
            extract_ppg_features(
                bvp_window
            )
        )

        features.update(
            extract_imu_features(
                acc_window
            )
        )

        # ----------------------------------------------------
        # Metadata
        # ----------------------------------------------------

        features["subject"] = subject_id
        features["window_start"] = start
        features["window_end"] = end
        features["label"] = label
        features["label_purity"] = purity

        rows.append(
            features
        )

    return pd.DataFrame(rows)

In [60]:
test_df = build_subject_dataset(
    "S2"
)

print(
    "Windows:",
    len(test_df)
)

print(
    "Features:",
    len(test_df.columns)
)

display(
    test_df.head()
)

Windows: 404
Features: 34


,eda_mean,eda_std,eda_min,eda_max,eda_range,eda_slope,scl_mean,phasic_mean,phasic_std,scr_count,...,acc_std,acc_energy,acc_magnitude_mean,acc_magnitude_std,acc_magnitude_energy,subject,window_start,window_end,label,label_purity
0,1.204161,0.111636,0.934426,1.311138,0.376712,2.259279e-03,1.209698,-0.005745,0.069136,2,...,36.270523,1350.644097,63.429250,5.353736,4051.932292,S2,0.0,30.0,0,1.0
1,1.290850,0.019784,1.212475,1.316263,0.103788,-3.487742e-08,1.289296,0.002162,0.016949,7,...,35.307884,1323.105903,62.931779,2.984783,3969.317708,S2,15.0,45.0,0,1.0
2,1.260221,0.080507,1.029344,1.616094,0.586750,-1.294824e-03,1.262088,-0.000119,0.053689,3,...,35.771924,1345.623611,63.289943,5.590523,4036.870833,S2,30.0,60.0,0,1.0
3,1.142074,0.129481,0.930582,1.616094,0.685512,-2.386095e-03,1.160894,-0.019716,0.086965,4,...,25.440784,1367.745833,63.540158,8.117006,4103.237500,S2,45.0,75.0,0,1.0
4,1.098844,0.076250,0.930582,1.239383,0.308801,1.708779e-03,1.101454,-0.001529,0.038536,6,...,25.082066,1343.948264,63.155621,6.573607,4031.844792,S2,60.0,90.0,0,1.0


In [61]:
test_df.isna().mean().sort_values(
    ascending=False
)

scr_recovery_mean       0.002475
eda_mean                0.000000
eda_std                 0.000000
eda_min                 0.000000
eda_range               0.000000
eda_max                 0.000000
scl_mean                0.000000
phasic_mean             0.000000
phasic_std              0.000000
eda_slope               0.000000
scr_count               0.000000
scr_amp_mean            0.000000
scr_rise_mean           0.000000
hr                      0.000000
hr_std                  0.000000
rmssd                   0.000000
sdnn                    0.000000
pnn50                   0.000000
ibi_mean                0.000000
ibi_std                 0.000000
ibi_min                 0.000000
ibi_max                 0.000000
peak_count              0.000000
acc_mean                0.000000
acc_std                 0.000000
acc_energy              0.000000
acc_magnitude_mean      0.000000
acc_magnitude_std       0.000000
acc_magnitude_energy    0.000000
subject                 0.000000
window_sta

In [62]:
# ============================================================
# BUILD SUBJECT-LEVEL BASELINES
# ============================================================

subject_baselines = []

for subject in subjects:

    print(
        f"Processing baseline: {subject}"
    )

    baseline_df = build_subject_dataset(
        subject,
        baseline_only=True
    )

    print(
        "  baseline windows:",
        len(baseline_df)
    )

    if len(baseline_df) == 0:
        continue

    feature_cols = [
        c for c in baseline_df.columns
        if c not in [
            "subject",
            "window_start",
            "window_end",
            "label",
            "label_purity"
        ]
    ]

    subject_baseline = (
        baseline_df[
            feature_cols
        ]
        .median()
    )

    subject_baseline[
        "subject"
    ] = subject

    subject_baselines.append(
        subject_baseline
    )

subject_baseline_df = pd.DataFrame(
    subject_baselines
)

print(
    "\nSubject baseline table:"
)

display(
    subject_baseline_df
)

Processing baseline: S10
  baseline windows: 78
Processing baseline: S11
  baseline windows: 77
Processing baseline: S13
  baseline windows: 77
Processing baseline: S14
  baseline windows: 77
Processing baseline: S15
  baseline windows: 76
Processing baseline: S16
  baseline windows: 77
Processing baseline: S17
  baseline windows: 78
Processing baseline: S2
  baseline windows: 74
Processing baseline: S3
  baseline windows: 74
Processing baseline: S4
  baseline windows: 76
Processing baseline: S5
  baseline windows: 78
Processing baseline: S6
  baseline windows: 77
Processing baseline: S7
  baseline windows: 77
Processing baseline: S8
  baseline windows: 76
Processing baseline: S9
  baseline windows: 77

Subject baseline table:


,eda_mean,eda_std,eda_min,eda_max,eda_range,eda_slope,scl_mean,phasic_mean,phasic_std,scr_count,...,ibi_min,ibi_max,peak_count,acc_mean,acc_std,acc_energy,acc_magnitude_mean,acc_magnitude_std,acc_magnitude_energy,subject
0,0.400819,0.004346,0.383752,0.425326,0.026863,0.000014,0.401076,0.000031,0.003192,21.0,...,390.6250,1351.5625,43.0,29.925521,20.852278,1333.884201,63.192758,2.118604,4001.652604,S10
1,2.957309,0.033624,2.862103,3.050153,0.134321,-0.000342,2.951990,-0.000027,0.015149,13.0,...,453.1250,1203.1250,36.0,-13.449653,34.654241,1391.941319,64.536087,2.421183,4175.823958,S11
2,4.510315,0.066714,4.137200,4.891418,0.309579,-0.000986,4.522026,-0.000406,0.035395,9.0,...,421.8750,1328.1250,40.0,21.273264,29.951952,1325.313542,62.996137,2.700823,3975.940625,S13
3,0.284079,0.002165,0.275305,0.293215,0.014071,0.000007,0.284288,-0.000057,0.001837,31.0,...,562.5000,1156.2500,36.0,-31.803819,19.665485,1396.508681,64.693865,1.517415,4189.526042,S14
4,0.279831,0.005770,0.267630,0.290017,0.026864,0.000024,0.279898,-0.000050,0.003315,20.5,...,421.8750,1523.4375,35.5,19.859028,30.295029,1311.522743,62.679858,2.088393,3934.568229,S15
5,0.285382,0.003910,0.276351,0.294263,0.020471,0.000003,0.285372,-0.000023,0.002960,20.0,...,421.8750,1390.6250,32.0,31.842361,17.967673,1339.391667,63.320772,2.214989,4018.175000,S16
6,1.282862,0.008499,1.264486,1.339972,0.041581,-0.000207,1.282951,-0.000138,0.003175,29.0,...,500.0000,1250.0000,32.0,-32.608333,18.480033,1396.451736,64.696135,0.626114,4189.355208,S17
7,0.143931,0.008042,0.134166,0.152746,0.029471,-0.000077,0.143929,0.000007,0.002531,18.0,...,414.0625,1250.0000,36.0,29.994444,21.003892,1341.425868,63.390571,1.501201,4024.277604,S2
8,1.032429,0.029777,0.949844,1.097188,0.136454,-0.000589,1.033658,-0.000177,0.010995,8.5,...,351.5625,1437.5000,30.0,-12.236111,35.113247,1387.248264,64.488455,1.104097,4161.744792,S3
9,0.121964,0.001757,0.115392,0.127545,0.011512,-0.000009,0.121960,0.000022,0.001571,29.0,...,359.3750,1328.1250,30.0,-31.239410,20.425968,1394.166146,64.647475,0.531794,4182.498438,S4


In [63]:
# ============================================================
# IMPORTANT PHYSIOLOGICAL BASELINE FEATURES
# ============================================================

important_features = [
    "hr",
    "hr_std",
    "rmssd",
    "sdnn",
    "pnn50",
    "ibi_mean",
    "ibi_std",
    "ibi_min",
    "ibi_max",
    "peak_count",
    "eda_mean",
    "eda_std",
    "scl_mean",
    "phasic_mean",
    "phasic_std",
    "scr_count",
    "scr_amp_mean",
    "scr_rise_mean",
    "scr_recovery_mean",
    "acc_magnitude_mean",
    "acc_magnitude_std"
]

display(
    subject_baseline_df[
        ["subject"] + important_features
    ].round(4)
)

,subject,hr,hr_std,rmssd,sdnn,pnn50,ibi_mean,ibi_std,ibi_min,ibi_max,...,eda_std,scl_mean,phasic_mean,phasic_std,scr_count,scr_amp_mean,scr_rise_mean,scr_recovery_mean,acc_magnitude_mean,acc_magnitude_std
0,S10,96.0000,20.9314,250.4430,210.1451,35.3906,683.1263,207.6575,390.6250,1351.5625,...,0.0043,0.4011,0.0000,0.0032,21.0,0.0049,0.4114,0.5000,63.1928,2.1186
1,S11,75.2941,16.8766,220.7656,155.7788,67.6471,811.1607,153.4003,453.1250,1203.1250,...,0.0336,2.9520,-0.0000,0.0151,13.0,0.0195,0.4375,0.6375,64.5361,2.4212
2,S13,85.3333,20.9113,273.1931,200.6272,56.4103,743.9904,198.1035,421.8750,1328.1250,...,0.0667,4.5220,-0.0004,0.0354,9.0,0.0516,0.5588,0.6500,62.9961,2.7008
3,S14,72.4528,11.7698,211.2012,138.4190,69.4444,832.7206,136.4829,562.5000,1156.2500,...,0.0022,0.2843,-0.0001,0.0018,31.0,0.0028,0.3952,0.5000,64.6939,1.5174
4,S15,80.0000,23.5409,362.6730,261.3970,68.3750,807.7567,257.7772,421.8750,1523.4375,...,0.0058,0.2799,-0.0001,0.0033,20.5,0.0038,0.4104,0.5000,62.6799,2.0884
5,S16,64.0000,22.1803,311.3550,228.0480,81.8182,934.3750,224.4565,421.8750,1390.6250,...,0.0039,0.2854,-0.0000,0.0030,20.0,0.0040,0.5000,0.5000,63.3208,2.2150
6,S17,65.0847,15.0125,224.1190,168.9829,82.7586,918.1735,166.0395,500.0000,1250.0000,...,0.0085,1.2830,-0.0001,0.0032,29.0,0.0053,0.3834,0.5833,64.6961,0.6261
7,S2,71.1111,21.7854,229.9232,174.2323,60.2431,824.2516,171.6486,414.0625,1250.0000,...,0.0080,0.1439,0.0000,0.0025,18.0,0.0028,0.5508,0.6726,63.3906,1.5012
8,S3,56.6874,28.5795,320.4097,264.4088,83.9355,1002.1552,259.5555,351.5625,1437.5000,...,0.0298,1.0337,-0.0002,0.0110,8.5,0.0174,0.6273,0.5670,64.4885,1.1041
9,S4,60.9524,23.3478,274.6129,198.7325,76.3333,972.9975,195.3064,359.3750,1328.1250,...,0.0018,0.1220,0.0000,0.0016,29.0,0.0026,0.4629,0.5000,64.6475,0.5318


In [64]:
# ============================================================
# VERIFY BASELINE LABELS
# ============================================================

for subject in subjects:

    df = build_subject_dataset(
        subject,
        baseline_only=True
    )

    if len(df) == 0:
        continue

    print(
        subject,
        "windows =", len(df),
        "| labels =",
        df["label"].unique(),
        "| purity min =",
        round(df["label_purity"].min(), 3),
        "| purity median =",
        round(df["label_purity"].median(), 3)
    )

S10 windows = 78 | labels = [1] | purity min = 0.905 | purity median = 1.0
S11 windows = 77 | labels = [1] | purity min = 1.0 | purity median = 1.0
S13 windows = 77 | labels = [1] | purity min = 1.0 | purity median = 1.0
S14 windows = 77 | labels = [1] | purity min = 1.0 | purity median = 1.0
S15 windows = 76 | labels = [1] | purity min = 1.0 | purity median = 1.0
S16 windows = 77 | labels = [1] | purity min = 1.0 | purity median = 1.0
S17 windows = 78 | labels = [1] | purity min = 0.926 | purity median = 1.0
S2 windows = 74 | labels = [1] | purity min = 1.0 | purity median = 1.0
S3 windows = 74 | labels = [1] | purity min = 1.0 | purity median = 1.0
S4 windows = 76 | labels = [1] | purity min = 0.99 | purity median = 1.0
S5 windows = 78 | labels = [1] | purity min = 1.0 | purity median = 1.0
S6 windows = 77 | labels = [1] | purity min = 1.0 | purity median = 1.0
S7 windows = 77 | labels = [1] | purity min = 1.0 | purity median = 1.0
S8 windows = 76 | labels = [1] | purity min = 1.0 | 

In [65]:
# ============================================================
# BASELINE DURATION CHECK
# ============================================================

for subject in subjects:

    data = load_subject(subject)

    labels = data["labels"]

    baseline_fraction = np.mean(
        labels == BASELINE_LABEL
    )

    total_duration = (
        len(labels) / LABEL_FS
    )

    baseline_duration = (
        np.sum(labels == BASELINE_LABEL)
        / LABEL_FS
    )

    print(
        f"{subject}: "
        f"total={total_duration/60:.2f} min, "
        f"baseline={baseline_duration/60:.2f} min, "
        f"fraction={baseline_fraction:.3f}"
    )

S10: total=91.60 min, baseline=19.67 min, fraction=0.215
S11: total=87.22 min, baseline=19.67 min, fraction=0.225
S13: total=92.28 min, baseline=19.67 min, fraction=0.213
S14: total=92.47 min, baseline=19.67 min, fraction=0.213
S15: total=87.53 min, baseline=19.58 min, fraction=0.224
S16: total=93.85 min, baseline=19.67 min, fraction=0.210
S17: total=98.67 min, baseline=19.68 min, fraction=0.199
S2: total=101.32 min, baseline=19.07 min, fraction=0.188
S3: total=108.22 min, baseline=19.00 min, fraction=0.176
S4: total=107.05 min, baseline=19.30 min, fraction=0.180
S5: total=104.30 min, baseline=19.97 min, fraction=0.191
S6: total=117.85 min, baseline=19.67 min, fraction=0.167
S7: total=87.30 min, baseline=19.77 min, fraction=0.226
S8: total=91.10 min, baseline=19.48 min, fraction=0.214
S9: total=87.05 min, baseline=19.67 min, fraction=0.226


In [66]:
# ============================================================
# UNIVERSAL WESAD BASELINE
# ============================================================

reference_features = [
    "hr",
    "hr_std",
    "rmssd",
    "sdnn",
    "pnn50",
    "ibi_mean",
    "ibi_std",
    "ibi_min",
    "ibi_max",

    "eda_mean",
    "eda_std",
    "eda_min",
    "eda_max",
    "eda_range",
    "eda_slope",
    "scl_mean",
    "phasic_mean",
    "phasic_std",
    "scr_count",
    "scr_amp_mean",
    "scr_rise_mean",
    "scr_recovery_mean",

    "acc_magnitude_mean",
    "acc_magnitude_std",
    "acc_magnitude_energy"
]

universal_baseline = {}

for feature in reference_features:

    values = (
        subject_baseline_df[feature]
        .dropna()
        .values
    )

    median = np.median(values)

    mad = median_abs_deviation(
        values,
        scale=1
    )

    universal_baseline[feature] = {
        "median": float(median),
        "mad": float(mad),
        "n_subjects": int(len(values))
    }

universal_baseline_df = pd.DataFrame(
    universal_baseline
).T

display(
    universal_baseline_df.round(5)
)

,median,mad,n_subjects
hr,71.11111,6.02637,15.0
hr_std,21.78541,1.56243,15.0
rmssd,273.19306,43.26988,15.0
sdnn,200.24456,30.42232,15.0
pnn50,68.37500,8.13194,15.0
ibi_mean,837.05357,32.36607,15.0
ibi_std,197.09090,28.81638,15.0
ibi_min,421.87500,31.25000,15.0
ibi_max,1328.12500,78.12500,15.0
eda_mean,0.40082,0.25689,15.0


In [67]:
display(
    subject_baseline_df[
        ["subject"] + important_features
    ].round(4)
)

,subject,hr,hr_std,rmssd,sdnn,pnn50,ibi_mean,ibi_std,ibi_min,ibi_max,...,eda_std,scl_mean,phasic_mean,phasic_std,scr_count,scr_amp_mean,scr_rise_mean,scr_recovery_mean,acc_magnitude_mean,acc_magnitude_std
0,S10,96.0000,20.9314,250.4430,210.1451,35.3906,683.1263,207.6575,390.6250,1351.5625,...,0.0043,0.4011,0.0000,0.0032,21.0,0.0049,0.4114,0.5000,63.1928,2.1186
1,S11,75.2941,16.8766,220.7656,155.7788,67.6471,811.1607,153.4003,453.1250,1203.1250,...,0.0336,2.9520,-0.0000,0.0151,13.0,0.0195,0.4375,0.6375,64.5361,2.4212
2,S13,85.3333,20.9113,273.1931,200.6272,56.4103,743.9904,198.1035,421.8750,1328.1250,...,0.0667,4.5220,-0.0004,0.0354,9.0,0.0516,0.5588,0.6500,62.9961,2.7008
3,S14,72.4528,11.7698,211.2012,138.4190,69.4444,832.7206,136.4829,562.5000,1156.2500,...,0.0022,0.2843,-0.0001,0.0018,31.0,0.0028,0.3952,0.5000,64.6939,1.5174
4,S15,80.0000,23.5409,362.6730,261.3970,68.3750,807.7567,257.7772,421.8750,1523.4375,...,0.0058,0.2799,-0.0001,0.0033,20.5,0.0038,0.4104,0.5000,62.6799,2.0884
5,S16,64.0000,22.1803,311.3550,228.0480,81.8182,934.3750,224.4565,421.8750,1390.6250,...,0.0039,0.2854,-0.0000,0.0030,20.0,0.0040,0.5000,0.5000,63.3208,2.2150
6,S17,65.0847,15.0125,224.1190,168.9829,82.7586,918.1735,166.0395,500.0000,1250.0000,...,0.0085,1.2830,-0.0001,0.0032,29.0,0.0053,0.3834,0.5833,64.6961,0.6261
7,S2,71.1111,21.7854,229.9232,174.2323,60.2431,824.2516,171.6486,414.0625,1250.0000,...,0.0080,0.1439,0.0000,0.0025,18.0,0.0028,0.5508,0.6726,63.3906,1.5012
8,S3,56.6874,28.5795,320.4097,264.4088,83.9355,1002.1552,259.5555,351.5625,1437.5000,...,0.0298,1.0337,-0.0002,0.0110,8.5,0.0174,0.6273,0.5670,64.4885,1.1041
9,S4,60.9524,23.3478,274.6129,198.7325,76.3333,972.9975,195.3064,359.3750,1328.1250,...,0.0018,0.1220,0.0000,0.0016,29.0,0.0026,0.4629,0.5000,64.6475,0.5318


In [68]:
# ============================================================
# VERIFY BASELINE LABELS AND PURITY
# ============================================================

for subject in subjects:

    df = build_subject_dataset(
        subject,
        baseline_only=True
    )

    if len(df) == 0:
        print(f"{subject}: NO BASELINE WINDOWS")
        continue

    unique_labels = df["label"].unique()

    min_purity = df["label_purity"].min()
    median_purity = df["label_purity"].median()
    max_purity = df["label_purity"].max()

    print(
        f"{subject}: "
        f"windows={len(df)}, "
        f"labels={unique_labels.tolist()}, "
        f"purity_min={min_purity:.3f}, "
        f"purity_median={median_purity:.3f}, "
        f"purity_max={max_purity:.3f}"
    )

S10: windows=78, labels=[1], purity_min=0.905, purity_median=1.000, purity_max=1.000
S11: windows=77, labels=[1], purity_min=1.000, purity_median=1.000, purity_max=1.000
S13: windows=77, labels=[1], purity_min=1.000, purity_median=1.000, purity_max=1.000
S14: windows=77, labels=[1], purity_min=1.000, purity_median=1.000, purity_max=1.000
S15: windows=76, labels=[1], purity_min=1.000, purity_median=1.000, purity_max=1.000
S16: windows=77, labels=[1], purity_min=1.000, purity_median=1.000, purity_max=1.000
S17: windows=78, labels=[1], purity_min=0.926, purity_median=1.000, purity_max=1.000
S2: windows=74, labels=[1], purity_min=1.000, purity_median=1.000, purity_max=1.000
S3: windows=74, labels=[1], purity_min=1.000, purity_median=1.000, purity_max=1.000
S4: windows=76, labels=[1], purity_min=0.990, purity_median=1.000, purity_max=1.000
S5: windows=78, labels=[1], purity_min=1.000, purity_median=1.000, purity_max=1.000
S6: windows=77, labels=[1], purity_min=1.000, purity_median=1.000, pu

In [69]:
# ============================================================
# INSPECT SUBJECT-LEVEL BASELINE FEATURES
# ============================================================

important_features = [
    "hr",
    "hr_std",
    "rmssd",
    "sdnn",
    "pnn50",
    "ibi_mean",
    "ibi_std",
    "ibi_min",
    "ibi_max",
    "peak_count",

    "eda_mean",
    "eda_std",
    "eda_min",
    "eda_max",
    "eda_range",
    "eda_slope",

    "scl_mean",
    "phasic_mean",
    "phasic_std",

    "scr_count",
    "scr_amp_mean",
    "scr_rise_mean",
    "scr_recovery_mean",

    "acc_magnitude_mean",
    "acc_magnitude_std",
    "acc_magnitude_energy"
]

available_features = [
    feature
    for feature in important_features
    if feature in subject_baseline_df.columns
]

display(
    subject_baseline_df[
        ["subject"] + available_features
    ].round(4)
)

,subject,hr,hr_std,rmssd,sdnn,pnn50,ibi_mean,ibi_std,ibi_min,ibi_max,...,scl_mean,phasic_mean,phasic_std,scr_count,scr_amp_mean,scr_rise_mean,scr_recovery_mean,acc_magnitude_mean,acc_magnitude_std,acc_magnitude_energy
0,S10,96.0000,20.9314,250.4430,210.1451,35.3906,683.1263,207.6575,390.6250,1351.5625,...,0.4011,0.0000,0.0032,21.0,0.0049,0.4114,0.5000,63.1928,2.1186,4001.6526
1,S11,75.2941,16.8766,220.7656,155.7788,67.6471,811.1607,153.4003,453.1250,1203.1250,...,2.9520,-0.0000,0.0151,13.0,0.0195,0.4375,0.6375,64.5361,2.4212,4175.8240
2,S13,85.3333,20.9113,273.1931,200.6272,56.4103,743.9904,198.1035,421.8750,1328.1250,...,4.5220,-0.0004,0.0354,9.0,0.0516,0.5588,0.6500,62.9961,2.7008,3975.9406
3,S14,72.4528,11.7698,211.2012,138.4190,69.4444,832.7206,136.4829,562.5000,1156.2500,...,0.2843,-0.0001,0.0018,31.0,0.0028,0.3952,0.5000,64.6939,1.5174,4189.5260
4,S15,80.0000,23.5409,362.6730,261.3970,68.3750,807.7567,257.7772,421.8750,1523.4375,...,0.2799,-0.0001,0.0033,20.5,0.0038,0.4104,0.5000,62.6799,2.0884,3934.5682
5,S16,64.0000,22.1803,311.3550,228.0480,81.8182,934.3750,224.4565,421.8750,1390.6250,...,0.2854,-0.0000,0.0030,20.0,0.0040,0.5000,0.5000,63.3208,2.2150,4018.1750
6,S17,65.0847,15.0125,224.1190,168.9829,82.7586,918.1735,166.0395,500.0000,1250.0000,...,1.2830,-0.0001,0.0032,29.0,0.0053,0.3834,0.5833,64.6961,0.6261,4189.3552
7,S2,71.1111,21.7854,229.9232,174.2323,60.2431,824.2516,171.6486,414.0625,1250.0000,...,0.1439,0.0000,0.0025,18.0,0.0028,0.5508,0.6726,63.3906,1.5012,4024.2776
8,S3,56.6874,28.5795,320.4097,264.4088,83.9355,1002.1552,259.5555,351.5625,1437.5000,...,1.0337,-0.0002,0.0110,8.5,0.0174,0.6273,0.5670,64.4885,1.1041,4161.7448
9,S4,60.9524,23.3478,274.6129,198.7325,76.3333,972.9975,195.3064,359.3750,1328.1250,...,0.1220,0.0000,0.0016,29.0,0.0026,0.4629,0.5000,64.6475,0.5318,4182.4984


In [70]:
# ============================================================
# CHECK ACTUAL ZERO VALUES
# ============================================================

baseline_feature_columns = [
    c
    for c in subject_baseline_df.columns
    if c != "subject"
]

zero_percentage = (
    (
        subject_baseline_df[
            baseline_feature_columns
        ] == 0
    )
    .mean()
    * 100
)

zero_percentage = zero_percentage.sort_values(
    ascending=False
)

display(
    zero_percentage.to_frame(
        name="zero_percentage"
    )
)

,zero_percentage
eda_mean,0.0
eda_std,0.0
eda_min,0.0
eda_max,0.0
eda_range,0.0
eda_slope,0.0
scl_mean,0.0
phasic_mean,0.0
phasic_std,0.0
scr_count,0.0


In [71]:
# ============================================================
# BASELINE STATISTICS ACROSS SUBJECTS
# ============================================================

display(
    subject_baseline_df[
        baseline_feature_columns
    ]
    .describe()
    .T
    .round(4)
)

,count,mean,std,min,25%,50%,75%,max
eda_mean,15.0,1.1845,1.5619,0.1220,0.2820,0.4008,1.1576,4.6901
eda_std,15.0,0.0153,0.0183,0.0018,0.0040,0.0080,0.0218,0.0667
eda_min,15.0,1.1322,1.4909,0.1154,0.2715,0.3838,1.1072,4.6087
eda_max,15.0,1.2395,1.6314,0.1275,0.2916,0.4253,1.2186,4.8914
eda_range,15.0,0.0679,0.0812,0.0115,0.0208,0.0295,0.0934,0.3096
eda_slope,15.0,-0.0002,0.0003,-0.0010,-0.0002,-0.0001,-0.0000,0.0000
scl_mean,15.0,1.1853,1.5633,0.1220,0.2821,0.4011,1.1583,4.6915
phasic_mean,15.0,-0.0001,0.0001,-0.0004,-0.0001,-0.0000,-0.0000,0.0000
phasic_std,15.0,0.0071,0.0092,0.0016,0.0024,0.0032,0.0072,0.0354
scr_count,15.0,20.8000,7.3553,8.5000,15.5000,21.0000,27.2500,31.0000


In [72]:
# ============================================================
# VERIFY WESAD BASELINE DURATION
# ============================================================

for subject in subjects:

    data = load_subject(subject)

    labels = data["labels"]

    total_duration_seconds = (
        len(labels) / LABEL_FS
    )

    baseline_samples = np.sum(
        labels == BASELINE_LABEL
    )

    baseline_duration_seconds = (
        baseline_samples / LABEL_FS
    )

    baseline_fraction = (
        baseline_samples /
        len(labels)
    )

    print(
        f"{subject}: "
        f"total={total_duration_seconds / 60:.2f} min, "
        f"baseline={baseline_duration_seconds / 60:.2f} min, "
        f"fraction={baseline_fraction:.3f}"
    )

S10: total=91.60 min, baseline=19.67 min, fraction=0.215
S11: total=87.22 min, baseline=19.67 min, fraction=0.225
S13: total=92.28 min, baseline=19.67 min, fraction=0.213
S14: total=92.47 min, baseline=19.67 min, fraction=0.213
S15: total=87.53 min, baseline=19.58 min, fraction=0.224
S16: total=93.85 min, baseline=19.67 min, fraction=0.210
S17: total=98.67 min, baseline=19.68 min, fraction=0.199
S2: total=101.32 min, baseline=19.07 min, fraction=0.188
S3: total=108.22 min, baseline=19.00 min, fraction=0.176
S4: total=107.05 min, baseline=19.30 min, fraction=0.180
S5: total=104.30 min, baseline=19.97 min, fraction=0.191
S6: total=117.85 min, baseline=19.67 min, fraction=0.167
S7: total=87.30 min, baseline=19.77 min, fraction=0.226
S8: total=91.10 min, baseline=19.48 min, fraction=0.214
S9: total=87.05 min, baseline=19.67 min, fraction=0.226


In [73]:
# ============================================================
# CHECK SUBJECT-LEVEL BASELINE FEATURES
# ============================================================

important_features = [
    "hr",
    "hr_std",
    "rmssd",
    "sdnn",
    "pnn50",
    "ibi_mean",
    "ibi_std",
    "ibi_min",
    "ibi_max",
    "peak_count",

    "eda_mean",
    "eda_std",
    "eda_min",
    "eda_max",
    "eda_range",
    "eda_slope",

    "scl_mean",
    "phasic_mean",
    "phasic_std",

    "scr_count",
    "scr_amp_mean",
    "scr_rise_mean",
    "scr_recovery_mean",

    "acc_magnitude_mean",
    "acc_magnitude_std",
    "acc_magnitude_energy"
]

available_features = [
    feature
    for feature in important_features
    if feature in subject_baseline_df.columns
]

display(
    subject_baseline_df[
        ["subject"] + available_features
    ].round(4)
)

,subject,hr,hr_std,rmssd,sdnn,pnn50,ibi_mean,ibi_std,ibi_min,ibi_max,...,scl_mean,phasic_mean,phasic_std,scr_count,scr_amp_mean,scr_rise_mean,scr_recovery_mean,acc_magnitude_mean,acc_magnitude_std,acc_magnitude_energy
0,S10,96.0000,20.9314,250.4430,210.1451,35.3906,683.1263,207.6575,390.6250,1351.5625,...,0.4011,0.0000,0.0032,21.0,0.0049,0.4114,0.5000,63.1928,2.1186,4001.6526
1,S11,75.2941,16.8766,220.7656,155.7788,67.6471,811.1607,153.4003,453.1250,1203.1250,...,2.9520,-0.0000,0.0151,13.0,0.0195,0.4375,0.6375,64.5361,2.4212,4175.8240
2,S13,85.3333,20.9113,273.1931,200.6272,56.4103,743.9904,198.1035,421.8750,1328.1250,...,4.5220,-0.0004,0.0354,9.0,0.0516,0.5588,0.6500,62.9961,2.7008,3975.9406
3,S14,72.4528,11.7698,211.2012,138.4190,69.4444,832.7206,136.4829,562.5000,1156.2500,...,0.2843,-0.0001,0.0018,31.0,0.0028,0.3952,0.5000,64.6939,1.5174,4189.5260
4,S15,80.0000,23.5409,362.6730,261.3970,68.3750,807.7567,257.7772,421.8750,1523.4375,...,0.2799,-0.0001,0.0033,20.5,0.0038,0.4104,0.5000,62.6799,2.0884,3934.5682
5,S16,64.0000,22.1803,311.3550,228.0480,81.8182,934.3750,224.4565,421.8750,1390.6250,...,0.2854,-0.0000,0.0030,20.0,0.0040,0.5000,0.5000,63.3208,2.2150,4018.1750
6,S17,65.0847,15.0125,224.1190,168.9829,82.7586,918.1735,166.0395,500.0000,1250.0000,...,1.2830,-0.0001,0.0032,29.0,0.0053,0.3834,0.5833,64.6961,0.6261,4189.3552
7,S2,71.1111,21.7854,229.9232,174.2323,60.2431,824.2516,171.6486,414.0625,1250.0000,...,0.1439,0.0000,0.0025,18.0,0.0028,0.5508,0.6726,63.3906,1.5012,4024.2776
8,S3,56.6874,28.5795,320.4097,264.4088,83.9355,1002.1552,259.5555,351.5625,1437.5000,...,1.0337,-0.0002,0.0110,8.5,0.0174,0.6273,0.5670,64.4885,1.1041,4161.7448
9,S4,60.9524,23.3478,274.6129,198.7325,76.3333,972.9975,195.3064,359.3750,1328.1250,...,0.1220,0.0000,0.0016,29.0,0.0026,0.4629,0.5000,64.6475,0.5318,4182.4984


In [74]:
# ============================================================
# CHECK ACTUAL ZERO VALUES
# ============================================================

baseline_feature_columns = [
    column
    for column in subject_baseline_df.columns
    if column != "subject"
]

zero_percentage = (
    (
        subject_baseline_df[
            baseline_feature_columns
        ] == 0
    )
    .mean()
    * 100
)

zero_percentage = zero_percentage.sort_values(
    ascending=False
)

display(
    zero_percentage.to_frame(
        name="zero_percentage"
    )
)

,zero_percentage
eda_mean,0.0
eda_std,0.0
eda_min,0.0
eda_max,0.0
eda_range,0.0
eda_slope,0.0
scl_mean,0.0
phasic_mean,0.0
phasic_std,0.0
scr_count,0.0


In [75]:
# ============================================================
# BASELINE STATISTICS ACROSS SUBJECTS
# ============================================================

display(
    subject_baseline_df[
        baseline_feature_columns
    ]
    .describe()
    .T
    .round(4)
)

,count,mean,std,min,25%,50%,75%,max
eda_mean,15.0,1.1845,1.5619,0.1220,0.2820,0.4008,1.1576,4.6901
eda_std,15.0,0.0153,0.0183,0.0018,0.0040,0.0080,0.0218,0.0667
eda_min,15.0,1.1322,1.4909,0.1154,0.2715,0.3838,1.1072,4.6087
eda_max,15.0,1.2395,1.6314,0.1275,0.2916,0.4253,1.2186,4.8914
eda_range,15.0,0.0679,0.0812,0.0115,0.0208,0.0295,0.0934,0.3096
eda_slope,15.0,-0.0002,0.0003,-0.0010,-0.0002,-0.0001,-0.0000,0.0000
scl_mean,15.0,1.1853,1.5633,0.1220,0.2821,0.4011,1.1583,4.6915
phasic_mean,15.0,-0.0001,0.0001,-0.0004,-0.0001,-0.0000,-0.0000,0.0000
phasic_std,15.0,0.0071,0.0092,0.0016,0.0024,0.0032,0.0072,0.0354
scr_count,15.0,20.8000,7.3553,8.5000,15.5000,21.0000,27.2500,31.0000


In [76]:
# ============================================================
# CREATE WESAD UNIVERSAL BASELINE
# ============================================================

import json
import numpy as np
from scipy.stats import median_abs_deviation

# ------------------------------------------------------------
# Features used for the universal physiological baseline
# ------------------------------------------------------------

reference_features = [
    # Heart / HRV
    "hr",
    "hr_std",
    "rmssd",
    "sdnn",
    "pnn50",
    "ibi_mean",
    "ibi_std",
    "ibi_min",
    "ibi_max",

    # EDA
    "eda_mean",
    "eda_std",
    "eda_min",
    "eda_max",
    "eda_range",
    "eda_slope",
    "scl_mean",
    "phasic_mean",
    "phasic_std",
    "scr_count",
    "scr_amp_mean",
    "scr_rise_mean",
    "scr_recovery_mean",

    # ACC magnitude
    "acc_magnitude_mean",
    "acc_magnitude_std",
    "acc_magnitude_energy"
]

# ------------------------------------------------------------
# Verify all required features exist
# ------------------------------------------------------------

missing_features = [
    feature
    for feature in reference_features
    if feature not in subject_baseline_df.columns
]

if len(missing_features) > 0:
    raise ValueError(
        f"Missing baseline features: {missing_features}"
    )

# ------------------------------------------------------------
# Build robust population reference
#
# Each subject contributes ONE baseline value.
# Therefore subjects with more windows do not dominate.
# ------------------------------------------------------------

universal_baseline = {}

for feature in reference_features:

    values = (
        subject_baseline_df[feature]
        .astype(float)
        .dropna()
        .values
    )

    if len(values) == 0:
        continue

    population_median = np.median(values)

    population_mad = median_abs_deviation(
        values,
        scale=1
    )

    universal_baseline[feature] = {
        "median": float(population_median),
        "mad": float(population_mad),
        "n_subjects": int(len(values))
    }

# ------------------------------------------------------------
# Convert to readable table
# ------------------------------------------------------------

universal_baseline_df = (
    pd.DataFrame(universal_baseline)
    .T
)

universal_baseline_df.index.name = "feature"

display(
    universal_baseline_df.round(5)
)

,median,mad,n_subjects
feature,,,
hr,71.11111,6.02637,15.0
hr_std,21.78541,1.56243,15.0
rmssd,273.19306,43.26988,15.0
sdnn,200.24456,30.42232,15.0
pnn50,68.37500,8.13194,15.0
ibi_mean,837.05357,32.36607,15.0
ibi_std,197.09090,28.81638,15.0
ibi_min,421.87500,31.25000,15.0
ibi_max,1328.12500,78.12500,15.0


In [77]:
# ============================================================
# SAVE WESAD UNIVERSAL BASELINE
# ============================================================

baseline_path = "/kaggle/working/wesad_universal_baseline.json"

with open(
    baseline_path,
    "w"
) as f:

    json.dump(
        universal_baseline,
        f,
        indent=4
    )

print(
    f"Universal baseline saved to:\n{baseline_path}"
)

Universal baseline saved to:
/kaggle/working/wesad_universal_baseline.json


In [78]:
# ============================================================
# VERIFY SAVED BASELINE FILE
# ============================================================

with open(
    baseline_path,
    "r"
) as f:

    loaded_baseline = json.load(f)

print(
    "Loaded features:",
    len(loaded_baseline)
)

print(
    "First 5 features:"
)

for i, (feature, values) in enumerate(
    loaded_baseline.items()
):

    print(
        feature,
        "->",
        values
    )

    if i >= 4:
        break

Loaded features: 25
First 5 features:
hr -> {'median': 71.11111111111111, 'mad': 6.026365348399253, 'n_subjects': 15}
hr_std -> {'median': 21.78541489737624, 'mad': 1.5624342349176032, 'n_subjects': 15}
rmssd -> {'median': 273.19306159203063, 'mad': 43.2698844960031, 'n_subjects': 15}
sdnn -> {'median': 200.2445571563455, 'mad': 30.42231686102886, 'n_subjects': 15}
pnn50 -> {'median': 68.375, 'mad': 8.131944444444443, 'n_subjects': 15}


In [79]:
# ============================================================
# UNIVERSAL BASELINE NORMALIZATION
# ============================================================

def calculate_universal_zscore(
    value,
    feature,
    universal_baseline,
    epsilon=1e-6
):
    """
    Calculate robust population-relative z-score.

    z = (value - population_median) /
        (1.4826 * population_MAD + epsilon)
    """

    if feature not in universal_baseline:
        return np.nan

    reference = universal_baseline[feature]

    median_value = reference["median"]
    mad_value = reference["mad"]

    robust_scale = (
        1.4826 * mad_value
    )

    denominator = (
        robust_scale + epsilon
    )

    z_score = (
        value - median_value
    ) / denominator

    return float(z_score)

In [80]:
# ============================================================
# TEST UNIVERSAL BASELINE NORMALIZATION
# ============================================================

test_features = [
    "hr",
    "rmssd",
    "sdnn",
    "pnn50",
    "eda_mean",
    "scr_count",
    "acc_magnitude_mean"
]

test_rows = []

for _, row in subject_baseline_df.iterrows():

    result = {
        "subject": row["subject"]
    }

    for feature in test_features:

        result[f"{feature}_raw"] = row[feature]

        result[f"{feature}_z"] = (
            calculate_universal_zscore(
                row[feature],
                feature,
                universal_baseline
            )
        )

    test_rows.append(result)

universal_z_test_df = pd.DataFrame(
    test_rows
)

display(
    universal_z_test_df.round(3)
)

,subject,hr_raw,hr_z,rmssd_raw,rmssd_z,sdnn_raw,sdnn_z,pnn50_raw,pnn50_z,eda_mean_raw,eda_mean_z,scr_count_raw,scr_count_z,acc_magnitude_mean_raw,acc_magnitude_mean_z
0,S10,96.000,2.786,250.443,-0.355,210.145,0.220,35.391,-2.736,0.401,0.000,21.0,0.000,63.193,-0.075
1,S11,75.294,0.468,220.766,-0.817,155.779,-0.986,67.647,-0.060,2.957,6.712,13.0,-0.771,64.536,2.630
2,S13,85.333,1.592,273.193,0.000,200.627,0.008,56.410,-0.992,4.510,10.790,9.0,-1.156,62.996,-0.471
3,S14,72.453,0.150,211.201,-0.966,138.419,-1.371,69.444,0.089,0.284,-0.307,31.0,0.964,64.694,2.948
4,S15,80.000,0.995,362.673,1.395,261.397,1.356,68.375,0.000,0.280,-0.318,20.5,-0.048,62.680,-1.109
5,S16,64.000,-0.796,311.355,0.595,228.048,0.616,81.818,1.115,0.285,-0.303,20.0,-0.096,63.321,0.182
6,S17,65.085,-0.674,224.119,-0.765,168.983,-0.693,82.759,1.193,1.283,2.316,29.0,0.771,64.696,2.952
7,S2,71.111,0.000,229.923,-0.674,174.232,-0.577,60.243,-0.674,0.144,-0.674,18.0,-0.289,63.391,0.323
8,S3,56.687,-1.614,320.410,0.736,264.409,1.423,83.935,1.291,1.032,1.658,8.5,-1.204,64.488,2.534
9,S4,60.952,-1.137,274.613,0.022,198.733,-0.034,76.333,0.660,0.122,-0.732,29.0,0.771,64.647,2.854


In [81]:
# ============================================================
# VERIFY UNIVERSAL NORMALIZATION
# ============================================================

verification_rows = []

for feature in reference_features:

    values = (
        subject_baseline_df[feature]
        .astype(float)
        .dropna()
        .values
    )

    z_values = [
        calculate_universal_zscore(
            value,
            feature,
            universal_baseline
        )
        for value in values
    ]

    verification_rows.append({
        "feature": feature,
        "z_mean": np.mean(z_values),
        "z_median": np.median(z_values),
        "z_min": np.min(z_values),
        "z_max": np.max(z_values)
    })

z_verification_df = pd.DataFrame(
    verification_rows
)

display(
    z_verification_df.round(4)
)

,feature,z_mean,z_median,z_min,z_max
0,hr,0.0738,0.0,-1.6143,2.7856
1,hr_std,-0.5018,0.0,-4.3237,2.9330
2,rmssd,-0.1846,0.0,-1.6432,1.3948
3,sdnn,-0.1080,0.0,-1.3707,1.4226
4,pnn50,-0.1932,0.0,-2.7358,1.2906
5,ibi_mean,0.3344,0.0,-3.2078,3.4406
6,ibi_std,-0.1107,0.0,-1.4186,1.4621
7,ibi_min,0.0787,0.0,-1.5176,3.0352
8,ibi_max,-0.1799,0.0,-1.4839,1.6862
9,eda_mean,2.0576,0.0,-0.7322,11.2621


In [82]:
# ============================================================
# INSPECT SCR RECOVERY BASELINE VALUES
# ============================================================

display(
    subject_baseline_df[
        [
            "subject",
            "scr_rise_mean",
            "scr_recovery_mean"
        ]
    ].round(6)
)

print("\nSCR recovery statistics:")

print(
    subject_baseline_df[
        "scr_recovery_mean"
    ].describe()
)

,subject,scr_rise_mean,scr_recovery_mean
0,S10,0.411442,0.500000
1,S11,0.437500,0.637500
2,S13,0.558824,0.650000
3,S14,0.395161,0.500000
4,S15,0.410428,0.500000
5,S16,0.500000,0.500000
6,S17,0.383363,0.583333
7,S2,0.550824,0.672619
8,S3,0.627315,0.566964
9,S4,0.462912,0.500000



SCR recovery statistics:
count    15.000000
mean      0.554583
std       0.069414
min       0.500000
25%       0.500000
50%       0.500000
75%       0.610417
max       0.672619
Name: scr_recovery_mean, dtype: float64


In [83]:
# ============================================================
# ROBUST EDA FEATURE EXTRACTION
# ============================================================

def extract_eda_features(eda_window):

    features = {
        "eda_mean": np.nan,
        "eda_std": np.nan,
        "eda_min": np.nan,
        "eda_max": np.nan,
        "eda_range": np.nan,
        "eda_slope": np.nan,

        "scl_mean": np.nan,
        "phasic_mean": np.nan,
        "phasic_std": np.nan,

        "scr_count": np.nan,
        "scr_amp_mean": np.nan,
        "scr_rise_mean": np.nan,
        "scr_recovery_mean": np.nan
    }

    try:

        eda_window = np.asarray(
            eda_window,
            dtype=float
        )

        if len(eda_window) < EDA_FS * 10:
            return features

        # ----------------------------------------------------
        # Remove invalid values
        # ----------------------------------------------------

        if not np.isfinite(eda_window).all():

            median_value = np.nanmedian(
                eda_window
            )

            eda_window = np.nan_to_num(
                eda_window,
                nan=median_value,
                posinf=median_value,
                neginf=median_value
            )

        # ----------------------------------------------------
        # Basic EDA cleaning
        # ----------------------------------------------------

        cleaned_eda = nk.eda_clean(
            eda_window,
            sampling_rate=EDA_FS,
            method="neurokit"
        )

        cleaned_eda = np.asarray(
            cleaned_eda,
            dtype=float
        )

        if len(cleaned_eda) == 0:
            return features

        # ----------------------------------------------------
        # Basic features
        # ----------------------------------------------------

        eda_mean = np.mean(
            cleaned_eda
        )

        eda_std = np.std(
            cleaned_eda,
            ddof=1
        )

        eda_min = np.min(
            cleaned_eda
        )

        eda_max = np.max(
            cleaned_eda
        )

        eda_range = (
            eda_max -
            eda_min
        )

        # ----------------------------------------------------
        # EDA slope
        # ----------------------------------------------------

        x = np.arange(
            len(cleaned_eda)
        )

        slope = np.polyfit(
            x,
            cleaned_eda,
            1
        )[0]

        # ----------------------------------------------------
        # Decompose EDA
        # ----------------------------------------------------

        eda_components = nk.eda_phasic(
            cleaned_eda,
            sampling_rate=EDA_FS
        )

        scl = np.asarray(
            eda_components["EDA_Tonic"],
            dtype=float
        )

        phasic = np.asarray(
            eda_components["EDA_Phasic"],
            dtype=float
        )

        scl_mean = np.mean(
            scl
        )

        phasic_mean = np.mean(
            phasic
        )

        phasic_std = np.std(
            phasic,
            ddof=1
        )

        # ----------------------------------------------------
        # SCR detection
        # ----------------------------------------------------

        try:

            signals, info = nk.eda_peaks(
                cleaned_eda,
                sampling_rate=EDA_FS
            )

        except Exception:

            signals = None
            info = {}

        # ----------------------------------------------------
        # SCR count
        # ----------------------------------------------------

        if (
            signals is not None
            and "SCR_Peaks" in signals.columns
        ):

            peak_indices = np.where(
                signals["SCR_Peaks"].values == 1
            )[0]

        else:

            peak_indices = np.array(
                [],
                dtype=int
            )

        scr_count = len(
            peak_indices
        )

        # ----------------------------------------------------
        # SCR amplitude
        # ----------------------------------------------------

        if (
            signals is not None
            and "SCR_Amplitude" in signals.columns
        ):

            scr_amplitudes = np.asarray(
                signals["SCR_Amplitude"],
                dtype=float
            )

            scr_amplitudes = scr_amplitudes[
                np.isfinite(scr_amplitudes)
                &
                (scr_amplitudes > 0)
            ]

        else:

            scr_amplitudes = np.array(
                [],
                dtype=float
            )

        if len(scr_amplitudes) > 0:

            scr_amp_mean = np.mean(
                scr_amplitudes
            )

        else:

            scr_amp_mean = 0.0

        # ----------------------------------------------------
        # SCR rise / recovery
        #
        # Use the actual NeuroKit SCR timing information
        # when available.
        # ----------------------------------------------------

        scr_rise_values = []
        scr_recovery_values = []

        if (
            signals is not None
            and len(peak_indices) > 0
        ):

            onset_indices = []

            if "SCR_Onsets" in signals.columns:

                onset_indices = np.where(
                    signals["SCR_Onsets"].values == 1
                )[0]

            half_recovery_indices = []

            if "SCR_Recovery" in signals.columns:

                half_recovery_indices = np.where(
                    signals["SCR_Recovery"].values == 1
                )[0]

            # ------------------------------------------------
            # Rise time
            # ------------------------------------------------

            if len(onset_indices) > 0:

                for peak_idx in peak_indices:

                    previous_onsets = (
                        onset_indices[
                            onset_indices <= peak_idx
                        ]
                    )

                    if len(previous_onsets) == 0:
                        continue

                    onset_idx = (
                        previous_onsets[-1]
                    )

                    rise_seconds = (
                        peak_idx -
                        onset_idx
                    ) / EDA_FS

                    # Physiologically reasonable SCR rise
                    if (
                        0 < rise_seconds <= 20
                    ):

                        scr_rise_values.append(
                            rise_seconds
                        )

            # ------------------------------------------------
            # Recovery time
            # ------------------------------------------------

            if len(half_recovery_indices) > 0:

                for peak_idx in peak_indices:

                    following_recoveries = (
                        half_recovery_indices[
                            half_recovery_indices > peak_idx
                        ]
                    )

                    if len(following_recoveries) == 0:
                        continue

                    recovery_idx = (
                        following_recoveries[0]
                    )

                    recovery_seconds = (
                        recovery_idx -
                        peak_idx
                    ) / EDA_FS

                    # Do not accept impossible recovery times
                    if (
                        0 < recovery_seconds <= 30
                    ):

                        scr_recovery_values.append(
                            recovery_seconds
                        )

        # ----------------------------------------------------
        # Final SCR features
        # ----------------------------------------------------

        if len(scr_rise_values) > 0:

            scr_rise_mean = np.mean(
                scr_rise_values
            )

        else:

            scr_rise_mean = 0.0

        if len(scr_recovery_values) > 0:

            scr_recovery_mean = np.mean(
                scr_recovery_values
            )

        else:

            scr_recovery_mean = 0.0

        # ----------------------------------------------------
        # Final sanity checks
        # ----------------------------------------------------

        values = {
            "eda_mean": eda_mean,
            "eda_std": eda_std,
            "eda_min": eda_min,
            "eda_max": eda_max,
            "eda_range": eda_range,
            "eda_slope": slope,

            "scl_mean": scl_mean,
            "phasic_mean": phasic_mean,
            "phasic_std": phasic_std,

            "scr_count": float(scr_count),
            "scr_amp_mean": scr_amp_mean,
            "scr_rise_mean": scr_rise_mean,
            "scr_recovery_mean": scr_recovery_mean
        }

        for key, value in values.items():

            if not np.isfinite(value):

                values[key] = np.nan

        return values

    except Exception:

        return features

In [84]:
# ============================================================
# REBUILD SUBJECT BASELINES AFTER EDA FIX
# ============================================================

subject_baselines = []

for subject in subjects:

    print(
        f"Processing baseline: {subject}"
    )

    baseline_df = build_subject_dataset(
        subject,
        baseline_only=True
    )

    print(
        "  baseline windows:",
        len(baseline_df)
    )

    if len(baseline_df) == 0:
        continue

    feature_cols = [
        column
        for column in baseline_df.columns
        if column not in [
            "subject",
            "window_start",
            "window_end",
            "label",
            "label_purity"
        ]
    ]

    subject_baseline = (
        baseline_df[
            feature_cols
        ]
        .median()
    )

    subject_baseline["subject"] = subject

    subject_baselines.append(
        subject_baseline
    )

subject_baseline_df = pd.DataFrame(
    subject_baselines
)

print(
    "\nSubjects with baseline:",
    len(subject_baseline_df)
)

display(
    subject_baseline_df.round(5)
)

Processing baseline: S10
  baseline windows: 78
Processing baseline: S11
  baseline windows: 77
Processing baseline: S13
  baseline windows: 77
Processing baseline: S14
  baseline windows: 77
Processing baseline: S15
  baseline windows: 76
Processing baseline: S16
  baseline windows: 77
Processing baseline: S17
  baseline windows: 78
Processing baseline: S2
  baseline windows: 74
Processing baseline: S3
  baseline windows: 74
Processing baseline: S4
  baseline windows: 76
Processing baseline: S5
  baseline windows: 78
Processing baseline: S6
  baseline windows: 77
Processing baseline: S7
  baseline windows: 77
Processing baseline: S8
  baseline windows: 76
Processing baseline: S9
  baseline windows: 77

Subjects with baseline: 15


,eda_mean,eda_std,eda_min,eda_max,eda_range,eda_slope,scl_mean,phasic_mean,phasic_std,scr_count,...,ibi_min,ibi_max,peak_count,acc_mean,acc_std,acc_energy,acc_magnitude_mean,acc_magnitude_std,acc_magnitude_energy,subject
0,0.40082,0.00436,0.38375,0.42533,0.02686,0.00001,0.40108,0.00003,0.00321,24.0,...,390.6250,1351.5625,43.0,29.92552,20.85228,1333.88420,63.19276,2.11860,4001.65260,S10
1,2.95731,0.03376,2.86210,3.05015,0.13432,-0.00034,2.95199,-0.00003,0.01521,13.0,...,453.1250,1203.1250,36.0,-13.44965,34.65424,1391.94132,64.53609,2.42118,4175.82396,S11
2,4.51031,0.06699,4.13720,4.89142,0.30958,-0.00099,4.52203,-0.00041,0.03554,8.0,...,421.8750,1328.1250,40.0,21.27326,29.95195,1325.31354,62.99614,2.70082,3975.94062,S13
3,0.28408,0.00217,0.27531,0.29322,0.01407,0.00001,0.28429,-0.00006,0.00185,31.0,...,562.5000,1156.2500,36.0,-31.80382,19.66549,1396.50868,64.69387,1.51742,4189.52604,S14
4,0.27983,0.00579,0.26763,0.29002,0.02686,0.00002,0.27990,-0.00005,0.00333,28.5,...,421.8750,1523.4375,35.5,19.85903,30.29503,1311.52274,62.67986,2.08839,3934.56823,S15
5,0.28538,0.00393,0.27635,0.29426,0.02047,0.00000,0.28537,-0.00002,0.00297,21.0,...,421.8750,1390.6250,32.0,31.84236,17.96767,1339.39167,63.32077,2.21499,4018.17500,S16
6,1.28286,0.00853,1.26449,1.33997,0.04158,-0.00021,1.28295,-0.00014,0.00319,30.0,...,500.0000,1250.0000,32.0,-32.60833,18.48003,1396.45174,64.69614,0.62611,4189.35521,S17
7,0.14393,0.00808,0.13417,0.15275,0.02947,-0.00008,0.14393,0.00001,0.00254,18.5,...,414.0625,1250.0000,36.0,29.99444,21.00389,1341.42587,63.39057,1.50120,4024.27760,S2
8,1.03243,0.02990,0.94984,1.09719,0.13645,-0.00059,1.03366,-0.00018,0.01104,8.0,...,351.5625,1437.5000,30.0,-12.23611,35.11325,1387.24826,64.48845,1.10410,4161.74479,S3
9,0.12196,0.00176,0.11539,0.12754,0.01151,-0.00001,0.12196,0.00002,0.00158,29.0,...,359.3750,1328.1250,30.0,-31.23941,20.42597,1394.16615,64.64748,0.53179,4182.49844,S4


In [85]:
# ============================================================
# VERIFY CORRECTED EDA FEATURES
# ============================================================

eda_check_features = [
    "eda_mean",
    "eda_std",
    "eda_range",
    "phasic_std",
    "scr_count",
    "scr_amp_mean",
    "scr_rise_mean",
    "scr_recovery_mean"
]

display(
    subject_baseline_df[
        ["subject"] + eda_check_features
    ].round(5)
)

print("\nStatistics:")

display(
    subject_baseline_df[
        eda_check_features
    ]
    .describe()
    .T
    .round(5)
)

,subject,eda_mean,eda_std,eda_range,phasic_std,scr_count,scr_amp_mean,scr_rise_mean,scr_recovery_mean
0,S10,0.40082,0.00436,0.02686,0.00321,24.0,0.00485,0.44349,3.96548
1,S11,2.95731,0.03376,0.13432,0.01521,13.0,0.02004,0.45833,1.75000
2,S13,4.51031,0.06699,0.30958,0.03554,8.0,0.04989,0.55556,1.72727
3,S14,0.28408,0.00217,0.01407,0.00185,31.0,0.00303,0.45833,1.31250
4,S15,0.27983,0.00579,0.02686,0.00333,28.5,0.00380,0.50000,2.82812
5,S16,0.28538,0.00393,0.02047,0.00297,21.0,0.00421,0.57500,4.10417
6,S17,1.28286,0.00853,0.04158,0.00319,30.0,0.00518,0.35774,3.01104
7,S2,0.14393,0.00808,0.02947,0.00254,18.5,0.00330,0.51356,1.44048
8,S3,1.03243,0.02990,0.13645,0.01104,8.0,0.01654,0.54904,1.28462
9,S4,0.12196,0.00176,0.01151,0.00158,29.0,0.00267,0.48077,0.00000



Statistics:


,count,mean,std,min,25%,50%,75%,max
eda_mean,15.0,1.18448,1.56186,0.12196,0.28195,0.40082,1.15765,4.69014
eda_std,15.0,0.01533,0.01839,0.00176,0.00405,0.00808,0.02190,0.06699
eda_range,15.0,0.06786,0.08115,0.01151,0.02079,0.02947,0.09338,0.30958
phasic_std,15.0,0.00714,0.00924,0.00158,0.00243,0.00319,0.00725,0.03554
scr_count,15.0,22.06667,7.80308,8.00000,18.25000,24.00000,28.75000,31.00000
scr_amp_mean,15.0,0.00951,0.01252,0.00267,0.00327,0.00421,0.01008,0.04989
scr_rise_mean,15.0,0.47742,0.05520,0.35774,0.45742,0.46212,0.50678,0.57500
scr_recovery_mean,15.0,2.37887,1.21538,0.00000,1.58387,2.50000,2.97165,4.40000


In [86]:
# ============================================================
# RECREATE UNIVERSAL BASELINE AFTER EDA FIX
# ============================================================

reference_features = [
    "hr",
    "hr_std",
    "rmssd",
    "sdnn",
    "pnn50",
    "ibi_mean",
    "ibi_std",
    "ibi_min",
    "ibi_max",

    "eda_mean",
    "eda_std",
    "eda_min",
    "eda_max",
    "eda_range",
    "eda_slope",
    "scl_mean",
    "phasic_mean",
    "phasic_std",
    "scr_count",
    "scr_amp_mean",
    "scr_rise_mean",
    "scr_recovery_mean",

    "acc_magnitude_mean",
    "acc_magnitude_std",
    "acc_magnitude_energy"
]

universal_baseline = {}

for feature in reference_features:

    values = (
        subject_baseline_df[feature]
        .astype(float)
        .dropna()
        .values
    )

    population_median = np.median(
        values
    )

    population_mad = median_abs_deviation(
        values,
        scale=1
    )

    universal_baseline[feature] = {
        "median": float(population_median),
        "mad": float(population_mad),
        "n_subjects": int(len(values))
    }

universal_baseline_df = (
    pd.DataFrame(
        universal_baseline
    ).T
)

universal_baseline_df.index.name = "feature"

display(
    universal_baseline_df.round(5)
)

# ------------------------------------------------------------
# Save updated baseline
# ------------------------------------------------------------

baseline_path = (
    "/kaggle/working/"
    "wesad_universal_baseline.json"
)

with open(
    baseline_path,
    "w"
) as f:

    json.dump(
        universal_baseline,
        f,
        indent=4
    )

print(
    f"\nSaved updated universal baseline:"
    f"\n{baseline_path}"
)

,median,mad,n_subjects
feature,,,
hr,71.11111,6.02637,15.0
hr_std,21.78541,1.56243,15.0
rmssd,273.19306,43.26988,15.0
sdnn,200.24456,30.42232,15.0
pnn50,68.37500,8.13194,15.0
ibi_mean,837.05357,32.36607,15.0
ibi_std,197.09090,28.81638,15.0
ibi_min,421.87500,31.25000,15.0
ibi_max,1328.12500,78.12500,15.0



Saved updated universal baseline:
/kaggle/working/wesad_universal_baseline.json


In [87]:
# ============================================================
# VALIDATE UPDATED UNIVERSAL BASELINE NORMALIZATION
# ============================================================

def calculate_universal_zscore(
    value,
    feature,
    universal_baseline,
    epsilon=1e-6
):
    if feature not in universal_baseline:
        return np.nan

    reference = universal_baseline[feature]

    median_value = reference["median"]
    mad_value = reference["mad"]

    robust_scale = 1.4826 * mad_value

    return float(
        (value - median_value) /
        (robust_scale + epsilon)
    )


# ------------------------------------------------------------
# Calculate z-scores for every WESAD subject baseline
# ------------------------------------------------------------

verification_rows = []

for _, row in subject_baseline_df.iterrows():

    result = {
        "subject": row["subject"]
    }

    for feature in reference_features:

        result[f"{feature}_z"] = (
            calculate_universal_zscore(
                row[feature],
                feature,
                universal_baseline
            )
        )

    verification_rows.append(result)


z_subject_df = pd.DataFrame(
    verification_rows
)


# ------------------------------------------------------------
# Summarize the z-score distributions
# ------------------------------------------------------------

z_summary_rows = []

for feature in reference_features:

    z_values = z_subject_df[
        f"{feature}_z"
    ].dropna()

    z_summary_rows.append({
        "feature": feature,
        "z_mean": np.mean(z_values),
        "z_median": np.median(z_values),
        "z_min": np.min(z_values),
        "z_max": np.max(z_values)
    })


z_verification_df = pd.DataFrame(
    z_summary_rows
)

display(
    z_verification_df.round(4)
)

,feature,z_mean,z_median,z_min,z_max
0,hr,0.0738,0.0,-1.6143,2.7856
1,hr_std,-0.5018,0.0,-4.3237,2.9330
2,rmssd,-0.1846,0.0,-1.6432,1.3948
3,sdnn,-0.1080,0.0,-1.3707,1.4226
4,pnn50,-0.1932,0.0,-2.7358,1.2906
5,ibi_mean,0.3344,0.0,-3.2078,3.4406
6,ibi_std,-0.1107,0.0,-1.4186,1.4621
7,ibi_min,0.0787,0.0,-1.5176,3.0352
8,ibi_max,-0.1799,0.0,-1.4839,1.6862
9,eda_mean,2.0576,0.0,-0.7322,11.2621


In [88]:
# ============================================================
# BUILD COMPLETE WESAD FEATURE DATASET
# ============================================================

all_subject_dfs = []

for subject in subjects:

    print(f"Processing subject: {subject}")

    subject_df = build_subject_dataset(
        subject,
        baseline_only=False
    )

    print(
        f"  windows: {len(subject_df)}"
    )

    if len(subject_df) == 0:
        continue

    all_subject_dfs.append(
        subject_df
    )


if len(all_subject_dfs) == 0:
    raise RuntimeError(
        "No subject datasets were created."
    )


wesad_df = pd.concat(
    all_subject_dfs,
    ignore_index=True
)


print("\n============================================================")
print("COMPLETE WESAD DATASET")
print("============================================================")

print(
    "Rows:",
    len(wesad_df)
)

print(
    "Subjects:",
    wesad_df["subject"].nunique()
)

print(
    "Features:",
    len([
        c for c in wesad_df.columns
        if c not in [
            "subject",
            "window_start",
            "window_end",
            "label",
            "label_purity"
        ]
    ])
)

display(
    wesad_df.head()
)

Processing subject: S10
  windows: 365
Processing subject: S11
  windows: 347
Processing subject: S13
  windows: 368
Processing subject: S14
  windows: 368
Processing subject: S15
  windows: 349
Processing subject: S16
  windows: 374
Processing subject: S17
  windows: 393
Processing subject: S2
  windows: 404
Processing subject: S3
  windows: 431
Processing subject: S4
  windows: 427
Processing subject: S5
  windows: 416
Processing subject: S6
  windows: 470
Processing subject: S7
  windows: 348
Processing subject: S8
  windows: 363
Processing subject: S9
  windows: 347

COMPLETE WESAD DATASET
Rows: 5770
Subjects: 15
Features: 29


,eda_mean,eda_std,eda_min,eda_max,eda_range,eda_slope,scl_mean,phasic_mean,phasic_std,scr_count,...,acc_std,acc_energy,acc_magnitude_mean,acc_magnitude_std,acc_magnitude_energy,subject,window_start,window_end,label,label_purity
0,0.349492,0.008502,0.331306,0.360727,0.029421,0.000222,0.350230,-0.000803,0.003492,33.0,...,36.697234,1357.830556,63.384151,7.479376,4073.491667,S10,0.0,30.0,0,1.0
1,0.359341,0.003780,0.349215,0.377356,0.028141,0.000091,0.359218,0.000068,0.002058,26.0,...,34.807721,1335.343056,63.216452,3.115981,4006.029167,S10,15.0,45.0,0,1.0
2,0.362966,0.002680,0.355610,0.377356,0.021746,0.000030,0.362754,0.000123,0.002337,28.0,...,35.978702,1327.841667,62.998276,3.839561,3983.525000,S10,30.0,60.0,0,1.0
3,0.361505,0.003709,0.353052,0.368402,0.015350,-0.000051,0.361360,0.000100,0.002707,21.0,...,33.656622,1325.939583,62.987627,3.221425,3977.818750,S10,45.0,75.0,0,1.0
4,0.359789,0.004399,0.345377,0.372240,0.026863,0.000053,0.359483,0.000150,0.003370,15.0,...,24.608775,1339.888889,63.292286,3.708529,4019.666667,S10,60.0,90.0,0,1.0


In [89]:
# ============================================================
# VERIFY COMPLETE WESAD DATASET
# ============================================================

print("Total windows:", len(wesad_df))
print("Subjects:", wesad_df["subject"].nunique())

feature_columns = [
    c for c in wesad_df.columns
    if c not in [
        "subject",
        "window_start",
        "window_end",
        "label",
        "label_purity"
    ]
]

print("Feature count:", len(feature_columns))

print("\nMissing values:")
print(
    wesad_df[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("\nLabel distribution:")
display(
    wesad_df["label"]
    .value_counts()
    .sort_index()
    .to_frame("window_count")
)

print("\nLabel purity:")
print(
    wesad_df["label_purity"].describe().round(4)
)

Total windows: 5770
Subjects: 15
Feature count: 29

Missing values:
eda_mean       0
eda_std        0
eda_min        0
eda_max        0
eda_range      0
eda_slope      0
scl_mean       0
phasic_mean    0
phasic_std     0
scr_count      0
dtype: int64

Label distribution:


,window_count
label,
0,2618
1,1173
2,667
3,371
4,785
5,51
6,53
7,52



Label purity:
count    5770.0000
mean        0.9794
std         0.0800
min         0.5031
25%         1.0000
50%         1.0000
75%         1.0000
max         1.0000
Name: label_purity, dtype: float64


In [90]:
# ============================================================
# CREATE NEUTRAL vs STRESS DATASET
# ============================================================

# WESAD:
# Label 1 = baseline / neutral
# Label 2 = stress

MODEL_MIN_PURITY = 0.90

wesad_binary_df = wesad_df[
    (wesad_df["label"].isin([1, 2])) &
    (wesad_df["label_purity"] >= MODEL_MIN_PURITY)
].copy()

# Binary target:
# 0 = Neutral
# 1 = Stress

wesad_binary_df["target"] = (
    wesad_binary_df["label"] == 2
).astype(int)

wesad_binary_df.reset_index(
    drop=True,
    inplace=True
)

print("============================================================")
print("WESAD BINARY DATASET")
print("============================================================")

print("Total windows:", len(wesad_binary_df))
print("Subjects:", wesad_binary_df["subject"].nunique())

print("\nOriginal labels:")
display(
    wesad_binary_df["label"]
    .value_counts()
    .sort_index()
    .to_frame("count")
)

print("\nBinary targets:")
display(
    wesad_binary_df["target"]
    .value_counts()
    .sort_index()
    .to_frame("count")
)

WESAD BINARY DATASET
Total windows: 1791
Subjects: 15

Original labels:


,count
label,
1,1149
2,642



Binary targets:


,count
target,
0,1149
1,642


In [91]:
# ============================================================
# FINAL UNIVERSAL BASELINE VALIDATION
# ============================================================

print("============================================================")
print("WESAD UNIVERSAL BASELINE")
print("============================================================")

print("Subjects used:", len(subject_baseline_df))
print("Features:", len(universal_baseline))

print("\nUniversal baseline:")

display(
    universal_baseline_df[
        ["median", "mad", "n_subjects"]
    ].round(5)
)

# Check every feature has all subjects
if (
    universal_baseline_df["n_subjects"] == 15
).all():
    print("\n✅ All 25 baseline features use all 15 subjects.")
else:
    print("\n⚠️ Some features have missing subjects.")

WESAD UNIVERSAL BASELINE
Subjects used: 15
Features: 25

Universal baseline:


,median,mad,n_subjects
feature,,,
hr,71.11111,6.02637,15.0
hr_std,21.78541,1.56243,15.0
rmssd,273.19306,43.26988,15.0
sdnn,200.24456,30.42232,15.0
pnn50,68.37500,8.13194,15.0
ibi_mean,837.05357,32.36607,15.0
ibi_std,197.09090,28.81638,15.0
ibi_min,421.87500,31.25000,15.0
ibi_max,1328.12500,78.12500,15.0



✅ All 25 baseline features use all 15 subjects.


In [92]:
# ============================================================
# FINAL SAVE
# ============================================================

import json

baseline_path = "/kaggle/working/wesad_universal_baseline.json"

with open(baseline_path, "w") as f:
    json.dump(
        universal_baseline,
        f,
        indent=4
    )

print(
    f"✅ Universal baseline saved:\n{baseline_path}"
)

✅ Universal baseline saved:
/kaggle/working/wesad_universal_baseline.json


In [93]:
# ============================================================
# FINAL UNIVERSAL BASELINE QUALITY CHECK
# ============================================================

print("============================================================")
print("FINAL UNIVERSAL BASELINE CHECK")
print("============================================================")

# Check number of subjects
assert len(subject_baseline_df) == 15, (
    f"Expected 15 subjects, found {len(subject_baseline_df)}"
)

# Check number of features
assert len(universal_baseline) == 25, (
    f"Expected 25 features, found {len(universal_baseline)}"
)

# Check subject count for every feature
assert (
    universal_baseline_df["n_subjects"] == 15
).all(), "Some features do not contain all 15 subjects."

# Check finite values
for feature in universal_baseline:

    median_value = universal_baseline[feature]["median"]
    mad_value = universal_baseline[feature]["mad"]

    assert np.isfinite(median_value), (
        f"Invalid median: {feature}"
    )

    assert np.isfinite(mad_value), (
        f"Invalid MAD: {feature}"
    )

print("✅ Subjects: 15")
print("✅ Baseline features: 25")
print("✅ Every feature uses all 15 subjects")
print("✅ No NaN/Inf in universal baseline")
print("✅ Universal baseline is valid")

FINAL UNIVERSAL BASELINE CHECK
✅ Subjects: 15
✅ Baseline features: 25
✅ Every feature uses all 15 subjects
✅ No NaN/Inf in universal baseline
✅ Universal baseline is valid


In [94]:
# ============================================================
# FINAL UNIVERSAL BASELINE TABLE
# ============================================================

final_baseline_table = universal_baseline_df[
    ["median", "mad", "n_subjects"]
].copy()

display(
    final_baseline_table.round(5)
)

,median,mad,n_subjects
feature,,,
hr,71.11111,6.02637,15.0
hr_std,21.78541,1.56243,15.0
rmssd,273.19306,43.26988,15.0
sdnn,200.24456,30.42232,15.0
pnn50,68.37500,8.13194,15.0
ibi_mean,837.05357,32.36607,15.0
ibi_std,197.09090,28.81638,15.0
ibi_min,421.87500,31.25000,15.0
ibi_max,1328.12500,78.12500,15.0


In [95]:
# ============================================================
# FINAL Z-SCORE SANITY CHECK
# ============================================================

z_check = []

for feature in reference_features:

    values = subject_baseline_df[
        feature
    ].astype(float)

    z_values = []

    for value in values:

        z = calculate_universal_zscore(
            value,
            feature,
            universal_baseline
        )

        z_values.append(z)

    z_check.append({
        "feature": feature,
        "median_z": np.median(z_values),
        "min_z": np.min(z_values),
        "max_z": np.max(z_values)
    })

z_check_df = pd.DataFrame(z_check)

display(
    z_check_df.round(4)
)

,feature,median_z,min_z,max_z
0,hr,0.0,-1.6143,2.7856
1,hr_std,0.0,-4.3237,2.9330
2,rmssd,0.0,-1.6432,1.3948
3,sdnn,0.0,-1.3707,1.4226
4,pnn50,0.0,-2.7358,1.2906
5,ibi_mean,0.0,-3.2078,3.4406
6,ibi_std,0.0,-1.4186,1.4621
7,ibi_min,0.0,-1.5176,3.0352
8,ibi_max,0.0,-1.4839,1.6862
9,eda_mean,0.0,-0.7322,11.2621


In [96]:
# ============================================================
# FINAL EXPORT
# ============================================================

import json
import os

baseline_path = (
    "/kaggle/working/"
    "wesad_universal_baseline.json"
)

with open(baseline_path, "w") as f:

    json.dump(
        universal_baseline,
        f,
        indent=4
    )

print("============================================================")
print("FINAL EXPORT")
print("============================================================")

print(
    "File:",
    baseline_path
)

print(
    "Exists:",
    os.path.exists(baseline_path)
)

print(
    "Size:",
    os.path.getsize(baseline_path),
    "bytes"
)

print("\n✅ WESAD UNIVERSAL BASELINE COMPLETE")

FINAL EXPORT
File: /kaggle/working/wesad_universal_baseline.json
Exists: True
Size: 3006 bytes

✅ WESAD UNIVERSAL BASELINE COMPLETE


In [97]:
# ============================================================
# SAVE BASELINE METADATA
# ============================================================

metadata = {
    "dataset": "WESAD",
    "subjects_used": 15,
    "baseline_label": 1,
    "baseline_selection": "WESAD neutral/baseline periods",
    "window_seconds": WINDOW_SECONDS,
    "step_seconds": STEP_SECONDS,
    "eda_sampling_rate": EDA_FS,
    "bvp_sampling_rate": BVP_FS,
    "acc_sampling_rate": ACC_FS,
    "baseline_features": len(reference_features),
    "aggregation": "subject median followed by population median",
    "population_scale": "MAD",
    "normalization": "robust population z-score",
    "personal_calibration_required": True,
    "baseline_update_during_session": False
}

metadata_path = (
    "/kaggle/working/"
    "wesad_baseline_metadata.json"
)

with open(metadata_path, "w") as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

print("✅ Metadata saved:")
print(metadata_path)

✅ Metadata saved:
/kaggle/working/wesad_baseline_metadata.json
